In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/model_ready_prices.csv")

print(df["Commodity"].unique())
print(df["Commodity"].value_counts())

<ArrowStringArray>
['Onion', 'Potato', 'Tomato', 'Wheat']
Length: 4, dtype: str
Commodity
Potato    22
Onion     21
Wheat      6
Tomato     3
Name: count, dtype: int64


In [2]:
import joblib
import os
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

feature_cols = ['Month', 'Year', 'Quarter', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_3', 'rolling_std_3']
target_col = 'Avg_Modal_Price'  # adjust if your actual target column is named differently

MIN_ROWS_FOR_SPLIT = 15  # below this, skip test split

results = {}

for crop in df['Commodity'].unique():
    crop_df = df[df['Commodity'] == crop].sort_values('Year')  # chronological order
    n = len(crop_df)
    
    X = crop_df[feature_cols]
    y = crop_df[target_col]

    if n >= MIN_ROWS_FOR_SPLIT:
        split_idx = int(n * 0.8)
        X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    else:
        X_train, y_train = X, y
        X_test, y_test = None, None  # not enough data to evaluate

    lr = LinearRegression().fit(X_train, y_train)
    rf = RandomForestRegressor(random_state=42).fit(X_train, y_train)

    joblib.dump(lr, f"{MODELS_DIR}/{crop.lower()}_linear_regression.pkl")
    joblib.dump(rf, f"{MODELS_DIR}/{crop.lower()}_random_forest.pkl")

    if X_test is not None and len(X_test) > 0:
        lr_pred = lr.predict(X_test)
        rf_pred = rf.predict(X_test)
        results[crop] = {
            "rows": n,
            "lr_mae": mean_absolute_error(y_test, lr_pred),
            "rf_mae": mean_absolute_error(y_test, rf_pred),
            "lr_r2": r2_score(y_test, lr_pred),
            "rf_r2": r2_score(y_test, rf_pred),
        }
    else:
        results[crop] = {"rows": n, "note": "insufficient data for evaluation"}

    print(f"✅ {crop}: {n} rows — models saved")

results


✅ Onion: 21 rows — models saved
✅ Potato: 22 rows — models saved
✅ Tomato: 3 rows — models saved
✅ Wheat: 6 rows — models saved


{'Onion': {'rows': 21,
  'lr_mae': 1475.9042122649716,
  'rf_mae': 479.1028002678213,
  'lr_r2': -11.414025411496485,
  'rf_r2': -0.15613266651941582},
 'Potato': {'rows': 22,
  'lr_mae': 758.2495320507138,
  'rf_mae': 195.9731689123874,
  'lr_r2': -278.9873220620596,
  'rf_r2': -32.70422488160029},
 'Tomato': {'rows': 3, 'note': 'insufficient data for evaluation'},
 'Wheat': {'rows': 6, 'note': 'insufficient data for evaluation'}}

In [3]:
print(df.columns.tolist())

['Commodity', 'Year', 'Month', 'Avg_Modal_Price', 'Date', 'Quarter', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_3', 'rolling_std_3']


In [4]:
import pandas as pd

summary_rows = []
for crop, metrics in results.items():
    if "note" in metrics:
        summary_rows.append({
            "Crop": crop, "Rows": metrics["rows"],
            "LR_MAE": None, "RF_MAE": None,
            "Best_Model": "N/A", "Note": metrics["note"]
        })
    else:
        best = "Random Forest" if metrics["rf_mae"] < metrics["lr_mae"] else "Linear Regression"
        summary_rows.append({
            "Crop": crop, "Rows": metrics["rows"],
            "LR_MAE": round(metrics["lr_mae"], 2),
            "RF_MAE": round(metrics["rf_mae"], 2),
            "Best_Model": best, "Note": ""
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("../data/processed/model_evaluation_summary.csv", index=False)
summary_df

,Crop,Rows,LR_MAE,RF_MAE,Best_Model,Note
0,Onion,21,1475.90,479.10,Random Forest,
1,Potato,22,758.25,195.97,Random Forest,
2,Tomato,3,NaN,NaN,N/A,insufficient data for evaluation
3,Wheat,6,NaN,NaN,N/A,insufficient data for evaluation


In [5]:
def predict_price(crop_name: str,
                  features: dict,
                  model_type: str = "random_forest") -> float:
    """
    Predict the crop price.
    """
    model = load_model(crop_name, model_type)
    input_df = pd.DataFrame([features])
    input_df = input_df[model.feature_names_in_]
    prediction = model.predict(input_df)[0]
    return round(float(prediction), 2)

In [6]:
import os

print(os.getcwd())

C:\Users\aswin\OneDrive\Desktop\crop-advisor\notebooks


In [7]:
import os

print(os.listdir(".."))

['.git', '.gitignore', '.ipynb_checkpoints', 'app.py', 'data', 'models', 'notebooks', 'plots', 'requirements.txt', 'src', 'venv']


In [8]:
import sys

sys.path.append("../src")

print(sys.path[-1])   # Should print ../src

../src


In [9]:
import importlib
import predictor

importlib.reload(predictor)

from predictor import predict_price

In [10]:
sample_features = {
    "lag_1": 1500,
    "lag_2": 1450,
    "lag_3": 1400,
    "rolling_mean_3": 1450,
    "rolling_std_3": 30,
    "Month": 6,
    "Year": 2025,
    "Quarter": 2
}

result = predict_price("Onion", sample_features)

print(f"Predicted Onion price: ₹{result}")

Predicted Onion price: ₹1983.31


In [11]:
import sys
sys.path.append("../src")
from predictor import predict_price

test_cases = {
    "onion": {"Month": 6, "Year": 2025, "Quarter": 2, "lag_1": 1500, "lag_2": 1450, "lag_3": 1400, "rolling_mean_3": 1450, "rolling_std_3": 30},
    "potato": {"Month": 6, "Year": 2025, "Quarter": 2, "lag_1": 900, "lag_2": 880, "lag_3": 870, "rolling_mean_3": 883, "rolling_std_3": 15},
}

for crop, features in test_cases.items():
    price = predict_price(crop, features)
    print(f"{crop.title()}: ₹{price}")

Onion: ₹1983.31
Potato: ₹1006.94


In [12]:
def get_latest_features(crop_name: str, df: pd.DataFrame) -> dict:
    """
    Auto-build the feature dict for a crop using its most recent rows,
    instead of the user typing in lag values manually.
    """
    crop_df = df[df['Commodity'].str.lower() == crop_name.lower()].sort_values('Year')
    latest = crop_df.iloc[-1]  # most recent row
    
    return {
        "Month": int(latest["Month"]),
        "Year": int(latest["Year"]),
        "Quarter": int(latest["Quarter"]),
        "lag_1": latest["lag_1"],
        "lag_2": latest["lag_2"],
        "lag_3": latest["lag_3"],
        "rolling_mean_3": latest["rolling_mean_3"],
        "rolling_std_3": latest["rolling_std_3"],
    }

In [13]:
def recommend_crop(df: pd.DataFrame, crops: list = None) -> pd.DataFrame:
    """
    Predict prices for all crops and rank by expected % price growth.
    """
    if crops is None:
        crops = df['Commodity'].unique()
    
    rows = []
    for crop in crops:
        try:
            features = get_latest_features(crop, df)
            predicted = predict_price(crop.lower(), features)
            recent_avg = features["rolling_mean_3"]
            pct_change = round(((predicted - recent_avg) / recent_avg) * 100, 2)
            rows.append({
                "Crop": crop,
                "Recent_Avg_Price": round(recent_avg, 2),
                "Predicted_Price": predicted,
                "Expected_Change_%": pct_change
            })
        except FileNotFoundError:
            rows.append({"Crop": crop, "Recent_Avg_Price": None,
                        "Predicted_Price": None, "Expected_Change_%": None})
    
    result_df = pd.DataFrame(rows).sort_values("Expected_Change_%", ascending=False)
    return result_df.reset_index(drop=True)

In [14]:
recommendation = recommend_crop(df)
recommendation

,Crop,Recent_Avg_Price,Predicted_Price,Expected_Change_%
0,Onion,1670.25,1857.66,11.22
1,Potato,1094.76,1201.40,9.74
2,Wheat,2463.70,2495.03,1.27
3,Tomato,2730.55,1462.76,-46.43


In [15]:
def recommend_crop(df: pd.DataFrame, crops: list = None, min_rows: int = 15) -> pd.DataFrame:
    if crops is None:
        crops = df['Commodity'].unique()
    
    rows = []
    for crop in crops:
        crop_row_count = len(df[df['Commodity'].str.lower() == crop.lower()])
        try:
            features = get_latest_features(crop, df)
            predicted = predict_price(crop.lower(), features)
            recent_avg = features["rolling_mean_3"]
            pct_change = round(((predicted - recent_avg) / recent_avg) * 100, 2)
            confidence = "High" if crop_row_count >= min_rows else "Low (limited data)"
            rows.append({
                "Crop": crop, "Recent_Avg_Price": round(recent_avg, 2),
                "Predicted_Price": predicted, "Expected_Change_%": pct_change,
                "Confidence": confidence
            })
        except FileNotFoundError:
            rows.append({"Crop": crop, "Recent_Avg_Price": None,
                        "Predicted_Price": None, "Expected_Change_%": None,
                        "Confidence": "N/A"})
    
    result_df = pd.DataFrame(rows).sort_values("Expected_Change_%", ascending=False)
    return result_df.reset_index(drop=True)

In [16]:
recommendation = recommend_crop(df)
recommendation

,Crop,Recent_Avg_Price,Predicted_Price,Expected_Change_%,Confidence
0,Onion,1670.25,1857.66,11.22,High
1,Potato,1094.76,1201.40,9.74,High
2,Wheat,2463.70,2495.03,1.27,Low (limited data)
3,Tomato,2730.55,1462.76,-46.43,Low (limited data)


In [17]:
 df.columns.tolist()


['Commodity',
 'Year',
 'Month',
 'Avg_Modal_Price',
 'Date',
 'Quarter',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_mean_3',
 'rolling_std_3']

In [18]:
import os
print(os.listdir("../models"))

['.ipynb_checkpoints', 'onion_linear_regression.pkl', 'onion_random_forest.pkl', 'potato_linear_regression.pkl', 'potato_random_forest.pkl', 'tomato_linear_regression.pkl', 'tomato_random_forest.pkl', 'wheat_linear_regression.pkl', 'wheat_random_forest.pkl']


In [19]:
print(df['rolling_mean_3'].describe())
print("Any zero values?", (df['rolling_mean_3'] == 0).sum())

count      52.000000
mean     2041.700458
std       888.823039
min       777.369818
25%      1402.595313
50%      1980.293585
75%      2427.531660
max      4497.171782
Name: rolling_mean_3, dtype: float64
Any zero values? 0


In [20]:
from predictor import predict_price
predict_price("rice", {"Month": 6, "Year": 2025, "Quarter": 2, "lag_1": 100, "lag_2": 100, "lag_3": 100, "rolling_mean_3": 100, "rolling_std_3": 5})

FileNotFoundError: Model not found:
C:\Users\aswin\OneDrive\Desktop\crop-advisor\src\..\models\rice_random_forest.pkl

In [21]:
predict_price("onion", {"Month": 6, "Year": 2025, "Quarter": 2, "lag_1": 1500, "lag_2": 1450, "lag_3": 1400, "rolling_mean_3": 1450})

ValueError: Missing required feature(s): ['rolling_std_3']

In [22]:
print(df['rolling_mean_3'].describe())
print("Any zero values?", (df['rolling_mean_3'] == 0).sum())

count      52.000000
mean     2041.700458
std       888.823039
min       777.369818
25%      1402.595313
50%      1980.293585
75%      2427.531660
max      4497.171782
Name: rolling_mean_3, dtype: float64
Any zero values? 0


In [23]:
len(df)

52

In [24]:
df['Commodity'].value_counts()

Commodity
Potato    22
Onion     21
Wheat      6
Tomato     3
Name: count, dtype: int64